# ⚡ Smart Power — Final Project (Day 1 Setup)

**Question:** When is electricity in the Netherlands cheapest **and** cleanest, and how much is driven by renewable generation? → a practical recommendation on the best hours to use power.

**This notebook = Day 1:** get the data sources working. Everything in English (this repo goes on GitHub / portfolio).

## ✅ Today's checklist

- [ ] Run **Data Source 1** (Open-Meteo) — it works if it prints values
- [ ] Turn it into a clean **pandas** table
- [ ] Run **Data Source 2** (EnergyZero electricity prices) — test it
- [ ] Register at transparency.entsoe.eu **+ email for API token** (arrives in up to 3 working days)
- [ ] Send project **scope to Adi + Miguel** on Slack
- [ ] Read the **5-minute pitch** out loud once (~5 min)

> Goal for today: scope is locked + at least one data source works + ENTSO-E token requested. That's it.

## Data Source 1 — Open-Meteo (renewables proxy, no API key)

Wind speed and solar radiation are a proxy for how much renewable energy is available.

In [1]:
import requests

# Amsterdam: wind speed + solar radiation (renewable-energy proxy)
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 52.37,
    "longitude": 4.90,
    "hourly": "wind_speed_10m,shortwave_radiation",
}

response = requests.get(url, params=params)
data = response.json()

# Quick check: first 5 hourly timestamps and wind values
print(data["hourly"]["time"][:5])
print(data["hourly"]["wind_speed_10m"][:5])

['2026-08-04T00:00', '2026-08-04T01:00', '2026-08-04T02:00', '2026-08-04T03:00', '2026-08-04T04:00']
[13.7, 2.5, 1.4, 4.0, 5.8]


In [2]:
import pandas as pd

# Turn the API response into a clean table
weather_df = pd.DataFrame({
    "timestamp": data["hourly"]["time"],
    "wind_speed": data["hourly"]["wind_speed_10m"],
    "solar_radiation": data["hourly"]["shortwave_radiation"],
})
weather_df["timestamp"] = pd.to_datetime(weather_df["timestamp"])

print(weather_df.shape)
weather_df.head()

(168, 3)


,timestamp,wind_speed,solar_radiation
0,2026-08-04 00:00:00,13.7,0.0
1,2026-08-04 01:00:00,2.5,0.0
2,2026-08-04 02:00:00,1.4,0.0
3,2026-08-04 03:00:00,4.0,0.0
4,2026-08-04 04:00:00,5.8,0.0


## Data Source 2 — Dutch electricity prices (EnergyZero, no API key)

EnergyZero exposes the hourly Dutch day-ahead electricity prices. No token needed.

> Run the cell and look at the printed keys first. If the field names are different from what we expect, tell me the printed structure and we'll adjust the parser.

In [3]:
from datetime import datetime, timedelta, timezone

# Use yesterday (a full day of prices is available)
day = (datetime.now(timezone.utc) - timedelta(days=1)).strftime("%Y-%m-%d")

price_url = "https://api.energyzero.nl/v1/energyprices"
price_params = {
    "fromDate": f"{day}T00:00:00.000Z",
    "tillDate": f"{day}T23:59:59.999Z",
    "interval": 4,       # hourly
    "usageType": 1,      # electricity
    "inclBtw": "true",   # include VAT
}

price_resp = requests.get(price_url, params=price_params)
price_data = price_resp.json()

# Defensive check: see the structure before parsing
print("Top-level keys:", list(price_data.keys()))
print("First price entry:", price_data.get("Prices", [{}])[0])

Top-level keys: ['Prices', 'intervalType', 'fromDate', 'tillDate', 'average']
First price entry: {'readingDate': '2026-08-03T00:00:00Z', 'price': 0.16}


In [4]:
# Build a clean price table (adjust keys if the printout above differs)
prices_df = pd.DataFrame(price_data["Prices"])
prices_df = prices_df.rename(columns={"readingDate": "timestamp", "price": "electricity_price"})
prices_df["timestamp"] = pd.to_datetime(prices_df["timestamp"])

print(prices_df.shape)
prices_df.head()

(24, 2)


,timestamp,electricity_price
0,2026-08-03 00:00:00+00:00,0.16
1,2026-08-03 01:00:00+00:00,0.16
2,2026-08-03 02:00:00+00:00,0.16
3,2026-08-03 03:00:00+00:00,0.17
4,2026-08-03 04:00:00+00:00,0.21


## Data Source 3 (upgrade) — ENTSO-E (needs token, requested today)

ENTSO-E gives richer price + actual generation-by-source data. It needs a security token,
which can take up to 3 working days. **We requested it today** — once it arrives we plug it in here.

Until then, the MVP runs fine on Open-Meteo + EnergyZero above.

## Next steps (this week)

1. **Join** prices + renewables on `timestamp` (one combined hourly table)
2. **EDA:** average electricity price by hour of day / day of week; correlation renewables ↔ price
3. **Add carbon** intensity (Electricity Maps API) → find cheapest + cleanest hours
4. **Dashboard** (Streamlit or Tableau) → "best hours to use electricity"
5. **Automate** the pipeline to run daily (GitHub Actions)

Keep the MVP small: one API + one clean table + one dashboard insight = a complete project.

In [5]:
!pip install entsoe-py

In [6]:
from entsoe import EntsoePandasClient
import pandas as pd

client = EntsoePandasClient(api_key="8d5a604d-bc89-48c9-800c-9874ec7dea72")

start = pd.Timestamp("20260801", tz="Europe/Amsterdam")
end   = pd.Timestamp("20260803", tz="Europe/Amsterdam")

# Day-ahead electricity prices for the Netherlands
prices = client.query_day_ahead_prices("NL", start=start, end=end)
print(prices.head())

# Actual generation per production type (renewable share!)
generation = client.query_generation("NL", start=start, end=end)
print(generation.head())

2026-08-01 00:00:00+02:00    182.99
2026-08-01 00:15:00+02:00    172.68
2026-08-01 00:30:00+02:00    163.31
2026-08-01 00:45:00+02:00    154.37
2026-08-01 01:00:00+02:00    168.13
dtype: float64
                                    Biomass                     \
                          Actual Aggregated Actual Consumption   
2026-08-01 00:00:00+02:00               0.0              8.504   
2026-08-01 00:15:00+02:00               0.0              8.012   
2026-08-01 00:30:00+02:00               0.0              7.874   
2026-08-01 00:45:00+02:00               0.0              8.234   
2026-08-01 01:00:00+02:00               0.0              8.182   

                                 Fossil Gas                     \
                          Actual Aggregated Actual Consumption   
2026-08-01 00:00:00+02:00          7213.906            176.423   
2026-08-01 00:15:00+02:00          7224.625            152.203   
2026-08-01 00:30:00+02:00          7232.589            152.209   
2026-08-01 0